# Backends

`Backend` is a small protocol: `sample`, `optimize`, `laplace`, each taking a `ModelSpec` and a
data dict — never a Python callable (review A2). `get_backend("laplace")` is always available;
`get_backend("numpyro")` returns a typed `Unsupported` when the extra is missing.

In [ ]:
import sys

import numpy as np

from axiom.core import D, Data, Likelihood, ModelSpec, Param, Prior, is_failure
from axiom.infer import BACKEND_NAMES, Backend, LaplaceBackend, NumpyroBackend, PointEstimate, Posterior, SampleSettings, get_backend

In [ ]:
print(BACKEND_NAMES)
lap = get_backend("laplace")
print(isinstance(lap, Backend), getattr(lap, "name", lap))
npr = get_backend("numpyro")
print(npr if is_failure(npr) else npr.name)
print("heavy modules loaded by importing axiom.infer:", [m for m in ("jax", "numpyro", "arviz") if m in sys.modules])

A conjugate normal–normal model to compare against the closed form.

In [ ]:
y = Data(name="y", dimension=D.outcome)
mu = Param(name="mu", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="fixed", hyper={"value": 1.0}))
model = ModelSpec(name="normal_normal", mean=mu, outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(mu, sigma))
rng = np.random.default_rng(0)
data = {"y": rng.normal(2.0, 1.0, 50)}
n, ybar = 50, data["y"].mean()
post_var = 1 / (1 / 100 + n)
print("closed form:", round(post_var * n * ybar, 4), "±", round(np.sqrt(post_var), 4))

In [ ]:
backend = LaplaceBackend()
pe: PointEstimate = backend.optimize(model, data, seed=0)
print(pe.method, pe.converged, round(float(pe.theta["mu"]), 4))
post = backend.laplace(model, data, draws=4000, seed=0)
if isinstance(post, Posterior):
    s = post.summary("mu")
    print(round(s.mean, 4), round(s.sd, 4), post.provenance["hessian_pd"], post.provenance["seed"])
else:
    print(post)

In [ ]:
settings = SampleSettings(draws=300, tune=300, chains=2)
if not is_failure(npr):
    nuts = NumpyroBackend()
    mc = nuts.sample(model, data, draws=settings.draws, tune=settings.tune, chains=settings.chains, seed=0)
    if isinstance(mc, Posterior):
        print(round(mc.summary("mu").mean, 3), mc.provenance["divergences"], mc.provenance["method"])